# Flux Scans

All experiments run with `flux=True` (or dedicated flux experiment classes).  
Set `qi` and `flux_gain` per section as needed.

# Setup

In [ ]:
cfg_file = 'sample_50_rfboard.yml'
expt_path = 'C:\\_Data\\sample_50_new\\'

qi = 2
flux_gain = 0.3  # Default flux gain for all scans below

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

np.set_printoptions(legacy="1.25")
from qick import QickConfig

from slab_qick_calib.exp_handling.instrumentmanager import InstrumentManager
import slab_qick_calib.experiments as meas
from slab_qick_calib.calib import qubit_tuning, measure_func
from slab_qick_calib.analysis import qubit_params
from slab_qick_calib.helpers import rfboard, qick_check, config, handy

%load_ext autoreload
%autoreload 2

handy.config_figs()

In [ ]:
cfg_path = os.path.join(os.getcwd(), '..', 'configs', cfg_file)
auto_cfg = config.load(cfg_path)

im = InstrumentManager(ns_address=auto_cfg['aliases']['ip'], port=8888)
print(im)

soc = im[auto_cfg['aliases']['soc']]
soccfg = QickConfig(soc.get_cfg())

cfg_dict = {'soc': soccfg, 'expt_path': expt_path, 'cfg_file': cfg_path, 'im': im}

In [ ]:
# Apply RF settings from config
rfboard.apply_config_rf_settings(soc, auto_cfg, cfg_file=cfg_path)
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)

# Resonator Spectroscopy (flux)

In [ ]:
update = False

auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
rspec = meas.ResSpec(cfg_dict, qi=qi, params={'span': 25, 'flux': True, 'flux_gain': flux_gain})

if update:
    rspec.update()
    auto_cfg = config.load(cfg_path)

## Resonator vs Flux (2D sweep)

In [ ]:
res_flux = meas.ResSpecFlux(cfg_dict, qi=qi)

# Qubit Spectroscopy (flux)

## Single scan with flux=True

In [ ]:
update = False

auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
params = {'length': 2, 'span': 100, 'gain': 0.002, 'flux': True, 'flux_gain': flux_gain}
qspec = meas.QubitSpec(cfg_dict, qi=qi, style='fine', params=params)

if update and qspec.status:
    auto_cfg = config.update_qubit(cfg_path, 'f_ge', qspec.data['best_fit'][2], qi)
    auto_cfg = config.update_qubit(cfg_path, 'kappa', 2 * qspec.data['best_fit'][3], qi)

## Qubit Spec Fast Flux (2D: freq vs flux gain)

In [ ]:
qsff = meas.QubitSpecFastFlux(cfg_dict, qi=qi, params={
    'gain_start': 0.3, 'gain_stop': -0.28, 'expts_gain': 40,
    'span': 150, 'expts': 200, 'reps': 2000, 'rounds': 1,
})

In [ ]:
# qsff.update()

# Rabi (flux)

## Amplitude Rabi

In [ ]:
update = False

auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
amp_rabi = meas.RabiFluxExperiment(
    cfg_dict, qi=qi,
    params={'flux_gain': flux_gain},
    disp_kwargs={'rescale': True},
)

if update and amp_rabi.status:
    config.update_qubit(cfg_path, ('pulses', 'pi_ge', 'gain'), amp_rabi.data['pi_length'], qi)

## Amplitude Rabi Chevron

In [ ]:
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
chevron = meas.RabiFluxChevronExperiment(
    cfg_dict, qi=qi,
    params={'flux_gain': flux_gain, 'span_f': 20, 'expts_f': 30},
)

## Length Rabi

In [ ]:
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
len_rabi = meas.RabiFluxExperiment(
    cfg_dict, qi=qi,
    params={'sweep': 'length', 'pulse_type': 'const', 'flux_gain': flux_gain},
    disp_kwargs={'rescale': True},
)

# Single Shot (flux)

In [ ]:
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
shot = meas.HistogramFluxExperiment(
    cfg_dict, qi=qi,
    params={'shots': 200000, 'flux_gain': flux_gain},
)
# shot.update(use_peak=False)

# T1 (flux)

In [ ]:
update = False

auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
t1 = meas.T1Experiment(
    cfg_dict, qi=qi,
    params={'flux': True, 'flux_gain': flux_gain, 'reps': 5500, 'expts': 250, 'rounds': 1, 'start': 0.025},
    disp_kwargs={'rescale': True},
)

if update:
    t1.update()

## T1 Fast Flux (2D: T1 vs flux gain)

In [ ]:
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
t1ff = meas.T1FastFlux(
    cfg_dict, qi=qi,
    params={'freq_span': 500, 'direction': 'neg', 'start_t1': 1.2, 'expts_gain': 150},
)

# T2 Ramsey (flux)

In [ ]:
update = False

auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
t2r = meas.T2Experiment(
    cfg_dict, qi=qi, max_err=10,
    params={'flux': True, 'flux_gain': flux_gain},
    disp_kwargs={'rescale': True},
)

if update and t2r.status:
    config.update_qubit(cfg_path, 'f_ge', t2r.data['new_freq'], qi)
    auto_cfg = config.update_qubit(cfg_path, 'T2r', t2r.data['best_fit'][3], qi, sig=2)

## T2 Echo (flux)

In [ ]:
update = False

auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
t2e = meas.T2Experiment(
    cfg_dict, qi=qi, max_err=10,
    params={'experiment_type': 'echo', 'flux': True, 'flux_gain': flux_gain, 'rounds': 3, 'start': 0.01},
    disp_kwargs={'rescale': True},
)

if update and t2e.status:
    auto_cfg = config.update_qubit(cfg_path, 'T2e', t2e.data['best_fit'][3], qi, sig=2)

## T2 Fast Flux (2D: T2 vs flux gain)

In [ ]:
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
t2ff = meas.T2FastFlux(
    cfg_dict, qi=qi,
    params={'freq_span': 500, 'direction': 'neg'},
)

# T1 Continuous (flux)

In [ ]:
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
t1cont = meas.T1ContFluxExperiment(
    cfg_dict, qi=qi,
    params={'flux_gain': flux_gain, 'shots': 120000},
)

In [ ]:
t1cont.display(filter_type='boxcar')